# APIs Lab

In this lab, you will work with different APIs to build datasets and perform basic analysis.

The goal is to practice how to:

- Make API requests  
- Explore JSON responses  
- Extract relevant information  
- Transform data into pandas DataFrames  
- Generate simple insights  

You will work like a data analyst using APIs data.

# Challenge 1: Pokémon Dataset & Analysis

## Goal

Build and analyze a Pokémon dataset using the API.

## Example (single request)

Below you can see how to get data for ONE Pokémon:

In [ ]:
#import requests

#url = "https://pokeapi.co/api/v2/pokemon/1"

#response = requests.get(url)
#data = response.json()

#data

## Tasks

1. Using the example above, get data for Pokémon IDs from 1 to 151

2. Extract:
   - name
   - height
   - weight
   - base_experience
   - first type

3. Create a DataFrame

4. Add a new column:
   - weight_kg (divide weight by 10)

5. Answer:
   - Which Pokémon is the heaviest?
   - What is the average weight?
   - How many Pokémon are there per type?

6. Answer:
   - Which type has the highest average weight?

In [1]:
import requests
import pandas as pd

pokemons = []
# Getting all the pokemons data from 1 to 151
for i in range(1, 152): 
    url = f"https://pokeapi.co/api/v2/pokemon/{i}"
    response = requests.get(url)
    data = response.json()

    pokemons.append(data)


In [2]:
df1 = []
## Extracting the data we require
for p in pokemons:
    pokemon_dict = {
        "name": p["name"],
        "height": p["height"],
        "weight": p["weight"],
        "base_experience": p["base_experience"],
        "type": p["types"][0]["type"]["name"]}
    
    df1.append(pokemon_dict)

In [3]:
## Creating the dataframe
df2 = pd.DataFrame(df1)
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151 entries, 0 to 150
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   name             151 non-null    object
 1   height           151 non-null    int64 
 2   weight           151 non-null    int64 
 3   base_experience  151 non-null    int64 
 4   type             151 non-null    object
dtypes: int64(3), object(2)
memory usage: 6.0+ KB


In [4]:
## Creating a new column with weight
df2["weight_kg"] = df2["weight"] / 10

In [5]:
## Heaviest pokemon is Snorlax
heavy = df2.loc[df2["weight_kg"].idxmax()]
heavy

name               snorlax
height                  21
weight                4600
base_experience        189
type                normal
weight_kg            460.0
Name: 142, dtype: object

In [6]:
## Average weight of the pokemons is 45 Kg
avg_w = df2["weight_kg"].mean()
print(avg_w)

45.951655629139076


In [7]:
## Pokemon per type
type_p = df2["type"].value_counts()
print(type_p)

type
water       28
normal      22
poison      14
grass       12
fire        12
bug         12
electric     9
rock         9
ground       8
psychic      8
fighting     7
ghost        3
dragon       3
fairy        2
ice          2
Name: count, dtype: int64


In [8]:
## Type with highest average weight
avg_w_2 = df2.groupby("type")["weight_kg"].mean()

type_heavy = avg_w_2.idxmax()
print(type_heavy, avg_w_2.max())

rock 87.61111111111111


# Challenge 2: Crypto Market Analysis

## Goal

Analyze cryptocurrency prices using real-time data.

## Tasks

1. Get prices for:
   - bitcoin
   - ethereum
   - solana
   - dogecoin

2. Convert the response into a DataFrame (coins as rows)

3. Rename columns properly

4. Add:
   - price_rank (ranking by price)
   - price_category:
        - "high" if > 1000  
        - "medium" if between 100 and 1000  
        - "low" if < 100  

5. Answer:
   - Which coin is the most expensive?
   - Which category has more coins?

6. **Bonus**:
   - Create a column with price difference vs bitcoin

In [9]:
# starter code

#import requests
#import pandas as pd

#url2 = "https://api.coingecko.com/api/v3/simple/price"

#params = {
    # complete this
#}

In [20]:
import requests
import pandas as pd
import numpy as np

url2 = "https://api.coingecko.com/api/v3/simple/price"

params = {"ids": "bitcoin,ethereum,solana,dogecoin",
          "vs_currencies": "usd"}

response = requests.get(url2, params=params)
data = response.json()

df3 = pd.DataFrame(data).T.reset_index()
df3.columns = ["coin", "price_usd"] # Renaming columns

df3["price_rank"] = df3["price_usd"].rank(ascending=False, method="dense").astype(int) # Add price rank: 1 = most expensive

# Add price category
def price_category(price):
    if price > 1000:
        return "high"
    elif 100 <= price <= 1000:
        return "medium"
    else:
        return "low"

df3["price_category"] = df3["price_usd"].apply(price_category)

# Bonus: Create a column with price difference vs bitcoin
bitcoin_price = df3.loc[df3["coin"] == "bitcoin", "price_usd"].iloc[0]
df3["vs_bitcoin"] = df3["price_usd"] - bitcoin_price

df3

,coin,price_usd,price_rank,price_category,vs_bitcoin
0,bitcoin,72359.000000,1,high,0.000000
1,dogecoin,0.091721,4,low,-72358.908279
2,ethereum,2229.960000,2,high,-70129.040000
3,solana,83.160000,3,low,-72275.840000


In [21]:

expensive = df3.loc[df3["price_usd"].idxmax(), "coin"] # Most Eepensive coin
print("Most expensive coin:", expensive)

largest = df3["price_category"].value_counts().idxmax() # Category with more coins ## However, both have the same counts, thus both should be considered #
print("Category with more coins:", largest)

Most expensive coin: bitcoin
Category with more coins: high


I made too many request and got error but the code is working without any problem !